In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
from dataclasses import dataclass,field
from typing import Optional
from copy import deepcopy

In [3]:
actions = ['STAY_OUT', 'PIT_SOFT', 'PIT_MEDIUM', 'PIT_HARD']

# Exact mapping from the models notebook: race -> (Hard/Medium/Soft) -> C1..C5
RACE_COMPOUND_TO_C = {
    'Bahrain': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
    'Saudi Arabia': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
    'Australia': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Japan': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
    'China': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
    'Miami': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
    'Emilia-Romagna': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Monaco': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Canada': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Spain': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
    'Austria': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Great Britain': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
    'Hungary': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Belgium': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
    'Netherlands': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
    'Italy': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Azerbaijan': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Singapore': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'USA (Austin)': {'Hard': 'C2', 'Medium': 'C3', 'Soft': 'C4'},
    'Mexico': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Brazil': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Las Vegas': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
    'Qatar': {'Hard': 'C1', 'Medium': 'C2', 'Soft': 'C3'},
    'Abu Dhabi': {'Hard': 'C3', 'Medium': 'C4', 'Soft': 'C5'},
}

# GP code (3 letters) -> race name used above
GP_CODE_TO_RACE = {
    'BAH': 'Bahrain',
    'SAU': 'Saudi Arabia',
    'AUS': 'Australia',
    'JPN': 'Japan',
    'CHN': 'China',
    'MIA': 'Miami',
    'EMI': 'Emilia-Romagna',
    'MON': 'Monaco',
    'CAN': 'Canada',
    'ESP': 'Spain',
    'AUT': 'Austria',
    'GBR': 'Great Britain',
    'HUN': 'Hungary',
    'BEL': 'Belgium',
    'NED': 'Netherlands',
    'ITA': 'Italy',
    'AZE': 'Azerbaijan',
    'SIN': 'Singapore',
    'USA': 'USA (Austin)',
    'MEX': 'Mexico',
    'BRA': 'Brazil',
    'LVS': 'Las Vegas',
    'QAT': 'Qatar',
    'ABU': 'Abu Dhabi',
}

C_HARDNESS = {'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4, 'C5': 5}

def get_compound_hardness(race_name: str, compound: str, default: Optional[int] = None) -> Optional[int]:
    """Return compound hardness in [1..5] using race name and compound."""
    if race_name is None or compound is None:
        return default

    race_map = RACE_COMPOUND_TO_C.get(str(race_name).strip())
    if race_map is None:
        return default

    c_compound = race_map.get(str(compound).strip().title())
    if c_compound is None:
        return default

    return C_HARDNESS.get(c_compound, default)

def get_compound_hardness_by_gp_code(gp_code: str, compound: str, default: Optional[int] = None) -> Optional[int]:
    """Return compound hardness in [1..5] using GP code (e.g. 'BAH', 'MON', 'ABU')."""
    if gp_code is None:
        return default

    race_name = GP_CODE_TO_RACE.get(str(gp_code).strip().upper())
    if race_name is None:
        return default

    return get_compound_hardness(race_name, compound, default)

In [20]:
# --- Normalisation des noms de GP pour les composés ---
RACE_NAME_TO_COMPOUND_KEY = {
    'Imola': 'Emilia-Romagna',
    'Baku': 'Azerbaijan',
    'Austin': 'USA (Austin)',
    'Melbourne': 'Australia',
}

def normalize_race_name_for_compound(race_name: Optional[str]) -> Optional[str]:
    if race_name is None:
        return None
    key = str(race_name).strip()
    return RACE_NAME_TO_COMPOUND_KEY.get(key, key)

def get_compound_hardness(race_name: str, compound: str, default: Optional[int] = None) -> Optional[int]:
    """Return compound hardness in [1..5] using race name and compound."""
    if race_name is None or compound is None:
        return default

    normalized_race = normalize_race_name_for_compound(race_name)
    race_map = RACE_COMPOUND_TO_C.get(normalized_race)
    if race_map is None:
        return default

    c_compound = race_map.get(str(compound).strip().title())
    if c_compound is None:
        return default

    return C_HARDNESS.get(c_compound, default)

def get_compound_hardness_by_gp_code(gp_code: str, compound: str, default: Optional[int] = None) -> Optional[int]:
    """Return compound hardness in [1..5] using GP code (e.g. 'BAH', 'MON', 'ABU')."""
    if gp_code is None:
        return default

    race_name = GP_CODE_TO_RACE.get(str(gp_code).strip().upper())
    if race_name is None:
        return default

    return get_compound_hardness(race_name, compound, default)

In [32]:
def get_tyre_life_pct(race_number, gp_id, compound_label, tyre_life, stint_length=None, compound_hardness=None):
    """Return tyre life percentage using TYRE_LIMITS_2024 and the current GP / compound."""
    gp_name = gp_id or RACE_NUMBER_TO_GP_ID.get(race_number)
    if gp_name is None:
        fallback_limit = stint_length if stint_length and stint_length > 0 else 30.0
        return float(tyre_life) / max(float(fallback_limit), 1.0)

    if 'normalize_race_name_for_compound' in globals():
        gp_name = normalize_race_name_for_compound(gp_name)

    if compound_label is None and compound_hardness is not None:
        race_map = RACE_COMPOUND_TO_C.get(gp_name, {})
        for label, c_code in race_map.items():
            if C_HARDNESS.get(c_code) == int(compound_hardness):
                compound_label = label.upper()
                break

    if compound_label is not None:
        compound_label = str(compound_label).strip().upper()

    tyre_limits = TYRE_LIMITS_2024.get(gp_name, {})
    limit = tyre_limits.get(compound_label) if compound_label else None

    if limit is None:
        limit = stint_length if stint_length and stint_length > 0 else 30.0

    return float(tyre_life) / max(float(limit), 1.0)


In [22]:
import importlib
import preprocess
import numpy as np

# 1. Force le rechargement du module
importlib.reload(preprocess)

# 2. Ré-importe la fonction spécifique
from preprocess import preprocess

# 3. (Optionnel) Vérification immédiate
import inspect
source = inspect.getsourcelines(preprocess)[0]
# On cherche si 'LapNumber' est bien présent dans le code chargé
has_lap_number = any("LapNumber" in line for line in source)

if has_lap_number:
    print("✅ Succès : La fonction preprocess a été mise à jour avec 'LapNumber'.")
else:
    print("⚠️ Attention : La fonction chargée ne contient toujours pas 'LapNumber'.")
    print("Vérifie que tu as bien SAUVEGARDÉ le fichier preprocess.py sur ton disque.")

✅ Succès : La fonction preprocess a été mise à jour avec 'LapNumber'.


In [ ]:
from race_state import (
    Pit_Time_MU,
    Pit_Time_SIGMA,
    RaceState,
    get_compound_hardness,
    get_compound_hardness_by_gp_code,
    get_tyre_life_pct,
)


In [34]:
# --- Alias GP pour la map stratégie (pit loss + wear factor) ---
GP_STRATEGY_ALIASES = {
    'Australia': 'Melbourne',
    'Emilia-Romagna': 'Imola',
    'Azerbaijan': 'Baku',
    'USA (Austin)': 'Austin',
}

for alias_name, strategy_name in GP_STRATEGY_ALIASES.items():
    if alias_name not in RaceState.GP_STRATEGY_DATA and strategy_name in RaceState.GP_STRATEGY_DATA:
        RaceState.GP_STRATEGY_DATA[alias_name] = dict(RaceState.GP_STRATEGY_DATA[strategy_name])

print('Alias GP stratégie appliqués:', GP_STRATEGY_ALIASES)

Alias GP stratégie appliqués: {'Australia': 'Melbourne', 'Emilia-Romagna': 'Imola', 'Azerbaijan': 'Baku', 'USA (Austin)': 'Austin'}


In [ ]:
import pandas as pd
import numpy as np
import pickle
from preprocess import preprocess

# --- 1. CHARGEMENT ---
with open('../Models.ipynb/models/pace_predictor_rf_final_partie2.pkl', 'rb') as f:
    rf_model = pickle.load(f)

df_raw = pd.read_csv('../classes/master_dataset_partie2_2024.csv')

# --- 2. EXTRACTION & TRI ---
event_name = 'Italian Grand Prix'
event_raw = df_raw[df_raw['Event'] == event_name].copy()

if event_raw.empty:
    print(f'Aucune ligne trouvée pour {event_name}.')
    state = None
else:
    event_raw = event_raw.sort_values(by=['Driver', 'LapNumber']).reset_index(drop=True)

    # --- 3. PREPROCESS ---
    mapping_driver = event_raw[['Driver']].copy()
    df_processed = preprocess(event_raw)

    if 'Driver' not in df_processed.columns:
        df_processed = df_processed.join(mapping_driver)

    if 'LapNumber' not in df_processed.columns:
        print('⚠️ LapNumber est manquant ! Vérification dans event_raw...')
        print(f"LapNumber dans event_raw ? {'LapNumber' in event_raw.columns}")

    # --- 4. PRÉPARATION DES TRAJECTOIRES ---
    if 'Driver' not in df_processed.columns:
        print('Impossible de reconstruire les pilotes après preprocess.')
        state = None
    else:
        driver_counts = df_processed.groupby('Driver').size().sort_values(ascending=False)

        if len(driver_counts) < 2:
            print('Impossible de construire la simulation: moins de 2 pilotes disponibles après preprocess.')
            state = None
        else:
            d1_id, d2_id = driver_counts.index[:2].tolist()
            d1_laps = df_processed[df_processed['Driver'] == d1_id].sort_values('LapNumber')
            d2_laps = df_processed[df_processed['Driver'] == d2_id].sort_values('LapNumber')

            print(f'Pilotes choisis: {d1_id} vs {d2_id}')
            print(f'Laps disponibles: {len(d1_laps)} vs {len(d2_laps)}')

            # --- 5. INITIALISATION DE LA SIMULATION ---
            target_idx = min(40, len(d1_laps) - 1, len(d2_laps) - 1)
            if target_idx < 0:
                print('Impossible d initialiser la simulation: index de tour invalide.')
                state = None
            else:
                state = RaceState.from_dataset_row(
                    driver_row=d1_laps.iloc[target_idx],
                    rival_row=d2_laps.iloc[target_idx],
                    total_laps=57,
                    gap=3.2
)

                print(f'Tour initial choisi : {state.lap} (index {target_idx})')
                print(f"TyreLife : {state.tyre_life}")
                print(f"Compound : {state.compound} (C{state.compound_hardness})")
                print(f"DeltaToBest : {state.delta_to_best:.3f}s")
                print(f"Gap rival : {state.gap_to_rival:.1f}s")

                # --- 6. TEST TRANSITION : STAY_OUT ---
                noise = np.random.normal(0, 0.217)
                new_state = state.transition('STAY_OUT', rf_model, noise)

                print('\nAprès STAY_OUT :')
                print(f"Tour : {new_state.lap}")
                print(f"TyreLife : {new_state.tyre_life}")
                print(f"DeltaToBest : {new_state.delta_to_best:.3f}s")
                print(f"Gap rival : {new_state.gap_to_rival:.1f}s")

                # --- 7. TEST TRANSITION : PIT_SOFT ---
                new_state_pit = state.transition('PIT_SOFT', rf_model, 0)

                print('\nAprès PIT_SOFT :')
                print(f"Tour : {new_state_pit.lap}")
                print(f"TyreLife : {new_state_pit.tyre_life}")
                print(f"Compound : {new_state_pit.compound}")
                print(f"Gap rival : {new_state_pit.gap_to_rival:.1f}s")

                # --- SIMULATION DE 10 TOURS : VERSTAPPEN VS LECLERC ---
                temp_state = state
                print(f"{'Tour':<6} | {'Pace (Delta)':<12} | {'Gap vs Max':<12} | {'TyreLife':<8}")
                print('-' * 45)

                for i in range(10):
                    noise = np.random.normal(0, 0.05)
                    temp_state = temp_state.transition('STAY_OUT', rf_model, noise)
                    print(f"{temp_state.lap:<6} | {temp_state.delta_to_best:>10.3f}s | {temp_state.gap_to_rival:>10.2f}s | {temp_state.tyre_life:<8}")

                # --- ANALYSE FINALE ---
                final_gap = temp_state.gap_to_rival
                if final_gap < 3.2:
                    print(f"\nRésultat : Leclerc revient ! L'écart est tombé à {final_gap:.2f}s.")
                else:
                    print(f"\nRésultat : Verstappen s'échappe... L'écart est monté à {final_gap:.2f}s.")

if 'state' not in globals() or state is None:
    print('Simulation non exécutée: données insuffisantes après preprocess pour ce GP.')

c:\Users\user\Desktop\personal\F1_strategy(ejjaw yebda hna ya zamil\Reinforcement_learning_partie\preprocess.py:252: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Team'] = team_series.fillna(team_fallback).fillna('Unknown')


Pilotes choisis: 1 vs 14
Laps disponibles: 46 vs 44
Tour initial choisi : 44 (index 40)
TyreLife : 37.0
Compound : UNKNOWN (C0)
DeltaToBest : 0.407s
Gap rival : 3.2s

Après STAY_OUT :
Tour : 45
TyreLife : 38.0
DeltaToBest : 0.444s
Gap rival : 2.9s

Après PIT_SOFT :
Tour : 45
TyreLife : 1
Compound : SOFT
Gap rival : -20.3s
Tour   | Pace (Delta) | Gap vs Max   | TyreLife
---------------------------------------------
45     |      0.364s |       2.96s | 38.0    
46     |      0.428s |       2.68s | 39.0    
47     |      0.382s |       2.70s | 40.0    
48     |      0.321s |       2.47s | 41.0    
49     |      0.329s |       2.32s | 42.0    
50     |      0.107s |       2.35s | 43.0    
51     |      0.194s |       2.31s | 44.0    
52     |      0.217s |       2.53s | 45.0    
53     |      0.181s |       2.43s | 46.0    
54     |      0.134s |       2.29s | 47.0    

Résultat : Leclerc revient ! L'écart est tombé à 2.29s.


In [ ]:
def run_fixed_strategy(initial_state, model, pit_lap, new_compound, noise=0.02):
    """Run one fixed pit strategy until the end of the race."""
    state = deepcopy(initial_state)
    history = []

    while state.lap < state.total_laps:
        action = 'STAY_OUT'
        if state.lap == pit_lap:
            action = f'PIT_{new_compound}'

        state = state.transition(action, model, noise=noise)
        history.append({
            'lap': state.lap,
            'gap': state.gap_to_rival,
            'tyre_life': state.tyre_life,
            'compound': state.compound,
            'action': action,
        })

    return state.gap_to_rival, history


def build_initial_state_for_gp(event_name, driver_rank_1=0, driver_rank_2=1, target_idx=None, gap=3.2, total_laps=57):
    """Build a RaceState from the two first available drivers of a given GP."""
    event_raw = df_raw[df_raw['Event'] == event_name].copy()
    if event_raw.empty:
        raise ValueError(f"Aucune donnée trouvée pour {event_name}.")

    event_raw = event_raw.sort_values(by=['Driver', 'LapNumber']).reset_index(drop=True)
    mapping_driver = event_raw[['Driver']].copy()
    df_processed_gp = preprocess(event_raw)

    if 'Driver' not in df_processed_gp.columns:
        df_processed_gp = df_processed_gp.join(mapping_driver)

    driver_counts = df_processed_gp.groupby('Driver').size().sort_values(ascending=False)
    if len(driver_counts) < 2:
        raise ValueError(f"Pas assez de pilotes disponibles après preprocess pour {event_name}.")

    driver_1, driver_2 = driver_counts.index[driver_rank_1], driver_counts.index[driver_rank_2]
    driver_1_laps = df_processed_gp[df_processed_gp['Driver'] == driver_1].sort_values('LapNumber')
    driver_2_laps = df_processed_gp[df_processed_gp['Driver'] == driver_2].sort_values('LapNumber')

    if target_idx is None:
        target_idx = min(40, len(driver_1_laps) - 1, len(driver_2_laps) - 1)

    if target_idx < 0:
        raise ValueError(f"Pas assez de tours disponibles pour initialiser {event_name}.")

    initial_state = RaceState.from_dataset_row(
        driver_row=driver_1_laps.iloc[target_idx],
        rival_row=driver_2_laps.iloc[target_idx],
        total_laps=total_laps,
        gap=gap,
    )
    return initial_state, driver_1, driver_2, target_idx, df_processed_gp


# --- Phase 4: Monte Carlo simple sur Bahrain 2024 ---
# Deux stratégies fixes pour valider le moteur de course avant le MCTS.
initial_state, driver_1, driver_2, start_idx, bahrain_df = build_initial_state_for_gp(
    'Bahrain Grand Prix',
    total_laps=57,
)

strategies = [
    {'label': 'S-M', 'pit_lap': 18, 'new_compound': 'MEDIUM'},
    {'label': 'S-H', 'pit_lap': 18, 'new_compound': 'HARD'},
]

results = []
histories = {}
for strategy in strategies:
    final_gap, history = run_fixed_strategy(
        initial_state=initial_state,
        model=rf_model,
        pit_lap=strategy['pit_lap'],
        new_compound=strategy['new_compound'],
        noise=0.02,
    )
    results.append({
        'strategy': strategy['label'],
        'pit_lap': strategy['pit_lap'],
        'new_compound': strategy['new_compound'],
        'final_gap': final_gap,
    })
    histories[strategy['label']] = pd.DataFrame(history)

results_df = pd.DataFrame(results).sort_values('final_gap').reset_index(drop=True)
print(f"GP simulé : Bahrain Grand Prix 2024")
print(f"Pilotes de départ : {driver_1} vs {driver_2}")
print(f"Tour initial : {initial_state.lap} / {initial_state.total_laps} (index preprocess = {start_idx})")
print()
print(results_df)

best_strategy = results_df.iloc[0]
print()
print(f"Meilleure stratégie simulée : {best_strategy['strategy']} -> gap final {best_strategy['final_gap']:.2f}s")

# Aperçu rapide de l'évolution de la meilleure stratégie
best_history = histories[best_strategy['strategy']]
print()
print(best_history.head(10))

c:\Users\user\Desktop\personal\F1_strategy(ejjaw yebda hna ya zamil\Reinforcement_learning_partie\preprocess.py:252: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Team'] = team_series.fillna(team_fallback).fillna('Unknown')


GP simulé : Bahrain Grand Prix 2024
Pilotes de départ : 14 vs 11
Tour initial : 46 / 57 (index preprocess = 40)

  strategy  pit_lap new_compound  final_gap
0      S-H       18         HARD  -0.282772
1      S-M       18       MEDIUM   0.574018

Meilleure stratégie simulée : S-H -> gap final -0.28s

   lap       gap  tyre_life compound    action
0   47  3.408688        6.0  UNKNOWN  STAY_OUT
1   48  3.484546        7.0  UNKNOWN  STAY_OUT
2   49  3.272160        8.0  UNKNOWN  STAY_OUT
3   50  3.150491        9.0  UNKNOWN  STAY_OUT
4   51  2.798972       10.0  UNKNOWN  STAY_OUT
5   52  2.324874       11.0  UNKNOWN  STAY_OUT
6   53  1.629142       12.0  UNKNOWN  STAY_OUT
7   54  0.924268       13.0  UNKNOWN  STAY_OUT
8   55  0.202083       14.0  UNKNOWN  STAY_OUT
9   56 -0.434675       15.0  UNKNOWN  STAY_OUT
